# Task 5 results: surrogates with error bars

Reads the committed tables in `results/`; no model is loaded. Scores compare surrogate predictions with numerical TGLF for shot 128913. `README.md` gives the commands that produced them.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.precision", 3)
HERE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks/ryan").is_dir()) / "notebooks/ryan/task5"
results = HERE / "results"

## Shared versus output-specific ensembles

Mean over three independently trained repeats, on the 1,100 test cases and the 12 profile cases. `separate` is the selected specialist per flux; `shared_reference` is the original joint baseline.

In [2]:
summary = pd.read_csv(results / "system_summary.csv")
summary.pivot_table(index="system", columns=["split", "target"], values="rmse_mean")

split                  profile_test                  test               
target                      Gamma_e    Q_e    Q_i Gamma_e     Q_e    Q_i
system                                                                  
capacity_control              0.088  0.864  1.127   0.253  14.365  1.212
output_specific_shared        0.088  1.213  1.065   0.251  13.662  1.213
separate                      0.071  0.660  1.068   0.226  13.138  1.088
separate_reference            0.073  0.744  1.216   0.233  13.666  1.096
shared                        0.088  1.213  1.048   0.251  13.662  1.483
shared_reference              0.110  1.758  1.297   0.290  14.098  1.253

Paired 95% case-bootstrap intervals for the RMSE difference (candidate minus reference) on the test set:

In [3]:
pd.read_csv(results / "paired_bootstrap.csv")[["target", "candidate", "reference", "mean_rmse_difference", "ci_95_low", "ci_95_high"]]

,target,candidate,reference,mean_rmse_difference,ci_95_low,ci_95_high
0,Q_e,separate,shared,-0.524,-2.617,1.464
1,Q_e,output_specific_shared,shared,0.000,0.000,0.000
2,Q_e,separate,capacity_control,-1.227,-2.509,0.523
3,Q_i,separate,shared,-0.395,-1.067,0.220
4,Q_i,output_specific_shared,shared,-0.270,-0.788,0.193
5,Q_i,separate,capacity_control,-0.125,-0.642,0.437
6,Gamma_e,separate,shared,-0.025,-0.050,0.005
7,Gamma_e,output_specific_shared,shared,0.000,0.000,0.000
8,Gamma_e,separate,capacity_control,-0.027,-0.051,-0.004


## Pooling 15 networks per flux

Compared with one five-network group chosen on validation.

In [4]:
pd.read_csv(results / "pooled_vs_five.csv")

,target,selected_by_validation,five_rmse,pooled_rmse,delta_rmse,delta_rmse_pct,ci95_low,ci95_high,five_r2,pooled_r2
0,Q_e,repeat_2,13.091,13.105,0.013,0.103,-0.141,0.274,0.880,0.880
1,Q_i,repeat_3,1.120,1.069,-0.050,-4.482,-0.140,0.051,0.993,0.994
2,Gamma_e,repeat_1,0.226,0.223,-0.003,-1.149,-0.008,0.002,0.976,0.977


## 90% prediction intervals

Coverage is the share of TGLF values inside the interval. `varied_test` holds the 240 intermediate-radius cases whose forecasts were frozen before TGLF was run.

In [5]:
scores = pd.read_csv(results / "interval_scores.csv")
scores.pivot_table(index=["split", "model"], columns="target", values=["coverage", "mean_width", "rmse"])

coverage               mean_width                 \
target                     Gamma_e    Q_e    Q_i    Gamma_e     Q_e    Q_i   
split           model                                                        
historical_test joint_5      0.924  0.907  0.913      0.589   9.561  4.131   
                pooled_15    0.926  0.909  0.913      0.337   3.974  1.620   
profile_test    joint_5      0.833  0.750  0.833      0.399   4.613  2.159   
                pooled_15    0.917  0.917  0.833      0.228   1.977  1.098   
validation_oof  joint_5      0.901  0.898  0.904      0.574   9.873  4.193   
                pooled_15    0.901  0.902  0.903      0.325   4.017  1.634   
varied_test     joint_5      0.883  0.912  0.838      0.559  10.160  3.428   
                pooled_15    0.858  0.821  0.846      0.359   4.410  1.674   

                             rmse                 
target                    Gamma_e     Q_e    Q_i  
split           model                             
historical_test joint_5     0.310  14.609  1.306  
                pooled_15   0.223  13.105  1.069  
profile_test    joint_5     0.108   1.296  1.326  
                pooled_15   0.069   0.635  1.064  
validation_oof  joint_5     0.278  14.325  1.428  
                pooled_15   0.192   9.439  0.773  
varied_test     joint_5     0.264   9.039  1.340  
                pooled_15   0.207   7.667  0.999

## Inference cost (one CPU process, two threads)

In [6]:
pd.read_csv(results / "system_cost_summary.csv")

,system,networks,parameters_total,inference_single_ms,inference_batch_ms,inference_batch_cases
0,shared,5,25775,0.384,1.915,1100
1,output_specific_shared,10,51550,0.761,3.657,1100
2,separate,15,182895,1.121,6.248,1100
3,shared_reference,5,25775,0.380,1.796,1100
4,separate_reference,15,77055,1.019,4.733,1100
5,capacity_control,5,182260,0.390,3.624,1100
